# Olist 电商数据分析项目

## 1 项目背景
Olist 是一个巴西电商平台公开数据集，包含订单、客户、商品、卖家、支付、评价、物流时间等多张业务表。

本项目聚焦于成交后的经营链路，重点分析订单履约时效、客户评分、支付金额、商品品类和地区表现之间的关系。

本项目的核心研究问题是：物流履约是否会影响客户满意度？如果存在影响，哪些订单、地区、品类或卖家更容易产生履约风险？平台可以如何通过预警和运营干预降低差评风险、提升用户体验？

第一阶段先对原始数据表进行数据理解和基础分析，明确每张表的粒度、主键、缺失值和重复情况，为后续清洗、建模和业务分析做准备。

## 2 数据表概览
本节观察每张表的行列数

In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data_raw")

print("当前工作目录：", Path.cwd())
print("数据目录：", DATA_DIR.resolve())

files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dfs = {}

for name, file in files.items():
    dfs[name] = pd.read_csv(DATA_DIR / file)
    # print(f'{name}:')
    # display(dfs[name].shape)
    # display(dfs[name].head())

table_overview = pd.DataFrame({
    "table_name": list(dfs.keys()),
    "rows": [df.shape[0] for df in dfs.values()],
    "columns": [df.shape[1] for df in dfs.values()]
})

display(table_overview)

当前工作目录： d:\olist-analysis\notebooks
数据目录： D:\olist-analysis\data_raw


,table_name,rows,columns
0,orders,99441,8
1,order_items,112650,7
2,payments,103886,5
3,reviews,99224,7
4,customers,99441,5
5,products,32951,9
6,sellers,3095,4
7,geolocation,1000163,5
8,category_translation,71,2


**观察与总结**

本数据集共有九张表，数据规模在十万行左右，其中orders是核心订单表；order_items则包含了订单对应的商品明细；payments为支付信息；reviews为商品评价；products和sellers分别为商品信息和卖家信息；geolocation为邮编地理位置；category_translation则为商品类目的英文翻译。

## 3 字段信息检查
本节先汇总各表字段清单，再重点检查订单、商品明细、支付等核心表的数据类型。

In [4]:
field_overview = []

for name, df in dfs.items():
    field_overview.append({
        "table": name,
        "columns_count": df.shape[1],
        "columns": ", ".join(df.columns.tolist())
    })

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

field_overview = pd.DataFrame(field_overview)
display(field_overview)

,table,columns_count,columns
0,orders,8,"order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date"
1,order_items,7,"order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value"
2,payments,5,"order_id, payment_sequential, payment_type, payment_installments, payment_value"
3,reviews,7,"review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp"
4,customers,5,"customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state"
5,products,9,"product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm"
6,sellers,4,"seller_id, seller_zip_code_prefix, seller_city, seller_state"
7,geolocation,5,"geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state"
8,category_translation,2,"product_category_name, product_category_name_english"


In [5]:
core_tables = ["orders", "order_items", "products", "customers", "payments", "reviews"]

for name in core_tables:
    print(f"\n===== {name} =====")
    dfs[name].info()


===== orders =====
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB

===== order_items =====
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null

**观察与总结**

通过字段清单和核心表 `.info()` 检查，可以初步看出该数据集包含订单、客户、商品、卖家、支付、评价、地理位置和品类翻译等多个业务主题，整体结构更接近关系型业务数据库，而不是单张销售明细表。

从字段类型看，订单表、评价表和订单商品明细表中包含多个时间字段，例如订单购买时间、审批时间、实际送达时间、预计送达时间、评价创建时间和发货截止时间。当前这些字段在读取后可能表现为 object 类型，后续需要统一转换为 datetime 类型，以支持配送时长、延迟天数等时间指标计算。

从字段内容看，orders 表包含订单状态和多个物流时间节点，是后续履约分析的核心表；reviews 表包含 review_score，是客户满意度分析的核心表；payments 表包含 payment_type、payment_installments 和 payment_value，可用于支付方式和支付金额分析；order_items 表包含 price、freight_value、product_id 和 seller_id，可用于商品、运费、卖家维度分析。

## 4 主键与表粒度检查

本节检查各表关键ID字段的唯一性，目的是明确每张表的一行代表什么，以及后续多表关联时是否可以直接JOIN。

In [6]:
def check_key(df: pd.DataFrame, key_cols: str | list):
    rows = df.shape[0]
    unique_rows = df[key_cols].drop_duplicates().shape[0]
    duplicated_rows = rows - unique_rows
    
    return rows, unique_rows, duplicated_rows, rows == unique_rows

key_candidates = [
    ("orders", ["order_id"], "订单主表：预期一行一个订单"),
    ("customers", ["customer_id"], "客户记录表：预期一行一个订单客户记录"),
    ("customers", ["customer_unique_id"], "真实用户标识：检查是否一行一个真实用户"),
    ("order_items", ["order_id"], "检查是否一行一个订单"),
    ("order_items", ["order_id", "order_item_id"], "订单商品明细：预期一行一个订单商品项"),
    ("payments", ["order_id"], "检查是否一行一个订单"),
    ("payments", ["order_id", "payment_sequential"], "支付明细：预期一行一个订单支付记录"),
    ("reviews", ["review_id"], "评价记录：预期一行一个评价"),
    ("reviews", ["order_id"], "检查是否一行一个订单评价"),
    ("products", ["product_id"], "商品表：预期一行一个商品"),
    ("sellers", ["seller_id"], "卖家表：预期一行一个卖家"),
    ("category_translation", ["product_category_name"], "类目翻译表：预期一行一个类目")
]

key_results = []

for table_name, key_cols, note in key_candidates:
    df = dfs[table_name]
    rows, unique_rows, duplicated_rows, is_unique = check_key(df, key_cols)
    
    key_results.append({
        "table": table_name,
        "candidate_key": " + ".join(key_cols),
        "business_meaning": note,
        "rows": rows,
        "unique_key_rows": unique_rows,
        "duplicated_key_rows": duplicated_rows,
        "is_unique": is_unique
    })

key_results = pd.DataFrame(key_results)
display(key_results)

,table,candidate_key,business_meaning,rows,unique_key_rows,duplicated_key_rows,is_unique
0,orders,order_id,订单主表：预期一行一个订单,99441,99441,0,True
1,customers,customer_id,客户记录表：预期一行一个订单客户记录,99441,99441,0,True
2,customers,customer_unique_id,真实用户标识：检查是否一行一个真实用户,99441,96096,3345,False
3,order_items,order_id,检查是否一行一个订单,112650,98666,13984,False
4,order_items,order_id + order_item_id,订单商品明细：预期一行一个订单商品项,112650,112650,0,True
5,payments,order_id,检查是否一行一个订单,103886,99440,4446,False
6,payments,order_id + payment_sequential,支付明细：预期一行一个订单支付记录,103886,103886,0,True
7,reviews,review_id,评价记录：预期一行一个评价,99224,98410,814,False
8,reviews,order_id,检查是否一行一个订单评价,99224,98673,551,False
9,products,product_id,商品表：预期一行一个商品,32951,32951,0,True


**观察与总结**

本节通过候选主键唯一性检查，确认各表的业务粒度以及后续多表关联方式。

orders表中order_id唯一，说明orders可以作为订单主表，粒度为“一行一个订单”。

customers表中customer_id唯一，但customer_unique_id不唯一。结合字段含义可以判断，customer_id更接近订单级客户记录ID，用于orders表与customers表之间的关联；customer_unique_id更接近真实用户的匿名标识，同一个customer_unique_id可能对应多条customer记录。因此，后续如果只是合并订单与客户地区信息，应使用customer_id；如果分析复购、Cohort或RFM，应使用customer_unique_id。

order_items表中order_id不唯一，但order_id+order_item_id可以唯一标识一行，说明该表粒度为“一行一个订单商品明细”。因此后续构建订单级宽表时，不能直接用order_items与orders合并后计算订单级指标，需要先按order_id聚合。

payments表中order_id不唯一，但order_id+payment_sequential可以唯一标识一行，说明该表粒度为“一行一个订单支付记录”。后续如果要分析订单支付金额，也需要先按order_id聚合。

reviews表中review_id可作为评价记录ID，但order_id存在少量重复，说明reviews表并不完全是一行一个订单评价。后续合并到订单级宽表前，需要先按order_id聚合评分。

products表的product_id唯一，sellers表的seller_id唯一，category_translation表的product_category_name唯一，这三张表可以作为维度表参与后续关联。

综上，后续构建订单级分析宽表时，应以orders表为主表。customers通过customer_id直接关联；order_items、payments、reviews需要先聚合到order_id粒度后再合并；products、sellers和category_translation可作为商品、卖家和品类维度表使用。

## 5 缺失值和重复值检测
本节主要检查表的数据质量，判断各个表有没有明显的缺失以及重复的问题

### 5.1 各表整体缺失和重复情况

In [7]:
missing_summary = []

for name, df in dfs.items():
    missing_summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": df.isna().sum().sum(),
        "missing_columns": ', '.join(df.columns[df.isna().sum()>0].tolist()),
        "missing_columns_count": (df.isna().sum() > 0).sum(),
        "duplicate_rows": df.duplicated().sum()
    })

missing_summary = pd.DataFrame(missing_summary)
display(missing_summary)

,table,rows,columns,missing_cells,missing_columns,missing_columns_count,duplicate_rows
0,orders,99441,8,4908,"order_approved_at, order_delivered_carrier_date, order_delivered_customer_date",3,0
1,order_items,112650,7,0,,0,0
2,payments,103886,5,0,,0,0
3,reviews,99224,7,145903,"review_comment_title, review_comment_message",2,0
4,customers,99441,5,0,,0,0
5,products,32951,9,2448,"product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm",8,0
6,sellers,3095,4,0,,0,0
7,geolocation,1000163,5,0,,0,261831
8,category_translation,71,2,0,,0,0


### 5.2 各表具体缺失字段

In [8]:
contain_null_dfs = ['orders', 'reviews', 'products']

for name in contain_null_dfs:
    df = dfs[name]
    missing_cols = df.isna().sum()
    missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
    
    print(f"\n===== {name} =====")
    display(missing_cols)


===== orders =====


order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64


===== reviews =====


review_comment_title      87656
review_comment_message    58247
dtype: int64


===== products =====


product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

### 5.3 关键缺失字段的业务影响判断

查看orders表的缺失字段分布

In [9]:
orders=dfs["orders"]

time_cols=[
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_missing_by_status=(
    orders[time_cols]
    .isna()
    .rename(columns=lambda col: f"{col}_missing")
    .assign(total_orders=1)
    .groupby(orders["order_status"])
    .sum()
)

display(orders_missing_by_status)

,order_approved_at_missing,order_delivered_carrier_date_missing,order_delivered_customer_date_missing,order_estimated_delivery_date_missing,total_orders
order_status,,,,,
approved,0,2,2,0,2
canceled,141,550,619,0,625
created,5,5,5,0,5
delivered,14,2,8,0,96478
invoiced,0,314,314,0,314
processing,0,301,301,0,301
shipped,0,0,1107,0,1107
unavailable,0,609,609,0,609


查看reviews表的缺失字段分布

In [10]:
reviews=dfs["reviews"]

review_missing=reviews[
    ["review_score","review_comment_title","review_comment_message"]
].isna().sum()

display(review_missing)

review_score                  0
review_comment_title      87656
review_comment_message    58247
dtype: int64

查看products表的缺失字段分布

In [11]:
products=dfs["products"]

product_missing=(
    products
    .isna()
    .assign(total_rows=1)
    .sum()
    .sort_values(ascending=False)
)

display(product_missing)

total_rows                    32951
product_category_name           610
product_photos_qty              610
product_name_lenght             610
product_description_lenght      610
product_weight_g                  2
product_height_cm                 2
product_length_cm                 2
product_width_cm                  2
product_id                        0
dtype: int64

**观察与总结**

本节对各表的缺失值和重复行进行了检查。从整行重复检查看，geolocation表存在较多重复记录，其他核心业务表未发现整行重复。但orders、reviews、products和geolocation表存在不同类型的数据质量问题，需要结合后续分析目标判断处理方式。

orders表中物流时间字段存在缺失。进一步按order_status分组检查后可以发现，部分缺失与订单未完成履约有关，例如取消、未送达或运输中的订单可能没有实际送达时间。由于本项目后续重点分析履约时效，因此后续计算配送天数和延迟天数时，应优先筛选order_status为delivered的订单，并继续检查已送达订单中的关键时间字段是否完整。

reviews表中review_comment_title和review_comment_message缺失较多，但review_score是本项目客户满意度分析的核心字段。由于很多用户可能只打分而不填写文字评论，因此评论文本缺失不直接影响第一阶段的评分分析。后续如果进行文本分析，则需要单独处理评论内容缺失问题。

products表中product_category_name及部分商品属性字段存在缺失，会影响后续品类维度分析。由于本项目核心主线是履约时效与客户评分，商品类目缺失暂不影响主线分析；但后续进行品类销售或品类差评分析时，需要将缺失类目标记为unknown或单独排除。

geolocation表存在较多整行重复，且该表粒度较复杂。由于第一阶段暂不使用经纬度信息，本阶段仅记录该问题，暂不进行清洗处理。

## 6基础业务分布检查

本节对订单状态、评价评分、支付方式和订单时间范围进行初步检查，用于了解数据集的基本业务分布。该部分不进行深入业务分析，只为后续确定分析范围和筛选规则提供依据。

订单状态分布

In [12]:
order_status_dist=(
    orders["order_status"]
    .value_counts()
    .to_frame("order_count")
)

order_status_dist["order_rate"]=(
    order_status_dist["order_count"]/order_status_dist["order_count"].sum()
).round(4)

display(order_status_dist)

,order_count,order_rate
order_status,,
delivered,96478,0.9702
shipped,1107,0.0111
canceled,625,0.0063
unavailable,609,0.0061
invoiced,314,0.0032
processing,301,0.0030
created,5,0.0001
approved,2,0.0000


评分分布

In [13]:
review_score_dist=(
    reviews["review_score"]
    .value_counts()
    .sort_index()
    .to_frame("review_count")
)

review_score_dist["review_rate"]=(
    review_score_dist["review_count"]/review_score_dist["review_count"].sum()
).round(4)

display(review_score_dist)

,review_count,review_rate
review_score,,
1,11424,0.1151
2,3151,0.0318
3,8179,0.0824
4,19142,0.1929
5,57328,0.5778


支付方式分布

In [14]:
payments=dfs["payments"]

payment_type_dist=(
    payments["payment_type"]
    .value_counts()
    .to_frame("payment_count")
)

payment_type_dist["payment_rate"]=(
    payment_type_dist["payment_count"]/payment_type_dist["payment_count"].sum()
).round(4)

display(payment_type_dist)

,payment_count,payment_rate
payment_type,,
credit_card,76795,0.7392
boleto,19784,0.1904
voucher,5775,0.0556
debit_card,1529,0.0147
not_defined,3,0.0000


订单时间分布

In [15]:
orders["order_purchase_timestamp"]=pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

print("最早下单时间:",orders["order_purchase_timestamp"].min())
print("最晚下单时间:",orders["order_purchase_timestamp"].max())

monthly_orders = (
    orders
    .query("order_status == 'delivered'")
    .assign(order_month=lambda x: x["order_purchase_timestamp"].dt.to_period("M"))
    .groupby("order_month")
    .size()
)

display(monthly_orders.head())
display(monthly_orders.tail())

最早下单时间: 2016-09-04 21:15:19
最晚下单时间: 2018-10-17 17:30:18


order_month
2016-09       1
2016-10     265
2016-12       1
2017-01     750
2017-02    1653
Freq: M, dtype: int64

order_month
2018-04    6798
2018-05    6749
2018-06    6099
2018-07    6159
2018-08    6351
Freq: M, dtype: int64

**观察与总结**

本节对订单状态、评价评分、支付方式和订单时间范围进行了基础业务分布检查。

从订单状态看，delivered订单占绝大多数，说明数据集中大部分订单已经完成履约。由于后续履约时效分析需要使用实际送达时间，因此后续计算配送天数、延迟天数和是否延迟时，应主要基于delivered订单展开。

从评价评分看，review_score覆盖1到5星，可作为客户满意度分析的核心字段。评分分布能够帮助后续进一步区分低分差评订单和高分满意订单，并分析履约时效是否与评分表现有关。

从支付方式看，数据集中存在credit_card、boleto、voucher、debit_card等支付类型。支付方式分布可作为后续支付行为和订单金额分析的基础。

从订单时间看，订单购买时间覆盖2016年9月至2018年10月，其中部分起止月份订单量较低，后续做时间趋势分析时需要注意不完整月份的影响。

综上，第6节的目标是建立对核心业务变量的初步认识，并为后续分析范围提供依据。后续正式分析将以delivered订单为主，围绕履约时效、客户评分、支付金额和订单明细进一步构建订单级分析宽表。